# EDGAR Financial Sentiment — Part 5: LLM-as-Classifier (Zero-/Few-Shot)

**Track B begins.** Parts 2–4 all *trained* something. Here we train **nothing**: we **prompt** `mistralai/Mistral-7B-Instruct-v0.3` (4-bit, same model as Part 4) to read a sentence and name its sentiment, zero-shot then few-shot, scored on the same Financial PhraseBank test set.

**What you'll practice (the core concept):** the prompting workflow — **(1)** constructing the classification prompt, **(2)** parsing the model's free-text reply into a label, **(3)** the inference/eval loop, and **(4)** discovering that *prompt wording is itself a lever* by running a head-to-head wording experiment. Those are your blanks.

> **Run in Google Colab with a T4 GPU.** Uses the same gated Mistral model as Part 4.

## 0. Why prompt a model to classify? — setting the stage

Every part so far **trained** something: Parts 2–3 fine-tuned full models, Part 4 trained LoRA adapters. All of that needs **labeled data, a GPU, and time**. But an instruction-tuned LLM can classify a sentence the moment it's downloaded — you just *ask* it. No training at all.

That makes Part 5 the project's **no-training baseline**, and it poses the question the whole project is really about:

> **Is fine-tuning worth it?** If simply prompting Mistral-7B lands within a couple of points of your fine-tuned FinBERT (~97%), the cheapest production choice might be to skip training entirely — no labels, no GPU, instant. If the gap is large, fine-tuning earns its cost. You can't decide until you've *measured* the baseline. So we build it.

**Two modes:** *zero-shot* (instruction only) and *few-shot* (a few labeled examples in the prompt — in-context learning, no weight updates).

**The mechanism.** We pose the task as a question and read the answer out of the generated text:
> *"Classify this financial sentence as negative, neutral, or positive. Reply with one word."*

Simple to run — the hard parts (and the real skills) are writing a prompt the model follows **reliably**, **parsing** a messy free-text reply back into a clean label, and realizing that the **exact wording** of the prompt changes the accuracy. You'll build all three, and measure that last one directly.

## 1. Setup

In [1]:
!pip install -q -U transformers bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.3 MB/s eta 0:00:00


In [2]:
import random, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset as HFDataset, ClassLabel

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

Device: cuda


### Hugging Face login (Mistral is gated)
Same as Part 4: accept the license at <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>, make a token, log in below.

In [4]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & seeds

In [5]:
torch.manual_seed(10); random.seed(10); np.random.seed(10)

MODEL_ID       = 'mistralai/Mistral-7B-Instruct-v0.3'
EVAL_N         = 150     # subsample of the test set (generation is slow); raise toward 453 for the full set
MAX_NEW_TOKENS = 8       # we only need one word back

LABELS       = ['negative', 'neutral', 'positive']   # the allowed answers (index 0/1/2)
LABEL_TO_IDX = {name: i for i, name in enumerate(LABELS)}

## 3. Load PhraseBank, subsample the test set, pick few-shot examples

Same data + stratified split as before. We evaluate on a ~`EVAL_N` slice (generation per example is slow), and pull **one few-shot example per class from the *train* split** — never the test set, to avoid leakage.

In [6]:
_URL = 'https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip'
_z = zipfile.ZipFile(io.BytesIO(requests.get(_URL, timeout=120).content))
_member = next(n for n in _z.namelist() if n.endswith('Sentences_AllAgree.txt'))
_raw = _z.read(_member).decode('latin-1')

_label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
_rows = []
for _line in _raw.splitlines():
    _line = _line.strip()
    if not _line:
        continue
    _sent, _lab = _line.rsplit('@', 1)
    _rows.append({'sentence': _sent, 'label': _label_map[_lab.strip()]})
_df = pd.DataFrame(_rows)

pb_full = HFDataset.from_pandas(_df).cast_column('label', ClassLabel(names=LABELS))
pb = pb_full.train_test_split(test_size=0.2, seed=10, stratify_by_column='label')
pb_train, pb_test = pb['train'], pb['test']
pb_eval = pb_test.shuffle(seed=10).select(range(min(EVAL_N, len(pb_test))))

FEWSHOT_EXAMPLES = []
for _cls in range(3):
    _ex = next(e for e in pb_train if e['label'] == _cls)
    FEWSHOT_EXAMPLES.append((_ex['sentence'], LABELS[_cls]))

print('Eval subsample:', len(pb_eval), 'sentences')
for s, lab in FEWSHOT_EXAMPLES:
    print(f'  [{lab}] {s[:80]}...')

Casting the dataset:   0%|          | 0/2264 [00:00<?, ? examples/s]

Eval subsample: 150 sentences
  [negative] Net profit in the three months through March 31 fell to (  x20ac ) 103 million (...
  [neutral] Sarantel , based in Wellingborough , UK , designs high-performance antennas for ...
  [positive] Sales improved to SEK 1,553 mn , compared with SEK 1,408 mn ....


## 4. Load Mistral-7B (4-bit) + a generation helper  *(provided)*

We load the model as a **causal LM** (text generator), not a classifier head — no training. `generate_reply` is the plumbing: wrap your prompt in the chat template, generate a few tokens **greedily** (deterministic), return just the new text.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.3


In [8]:
import torch

@torch.no_grad()
def generate_reply(prompt_text, max_new_tokens=MAX_NEW_TOKENS):
    """Send one user message through the chat template and return the model's reply text."""
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)

    # Extract input_ids and attention_mask from the BatchEncoding object
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    out = model.generate(input_ids=input_ids, attention_mask=attention_mask,
                         max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

    # Use input_ids.shape[-1] for slicing the newly generated tokens
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 5. Build the prompt  &larr; **your code**

Write `build_prompt(sentence, examples=())` returning the **user-message string**:
1. an **instruction** naming the three labels and asking for **one word only**;
2. any **few-shot examples** (each an `(sentence, label)` pair) in the same format you'll ask the target in;
3. the target sentence ending with a `Sentiment:` cue.

The same function serves both modes — zero-shot passes `examples=()`, few-shot passes the list.

In [9]:
def build_prompt(sentence, examples=()):
    """Return the user-message text for one classification (zero- or few-shot)."""
    ### YOUR CODE HERE
    # 1. start with an instruction: name the 3 labels (negative/neutral/positive) and say
    #    to reply with ONLY that one word.
    # 2. for each (ex_sentence, ex_label) in examples, append the SAME format you'll ask the
    #    target in (e.g.  Sentence: "<ex_sentence>"  /  Sentiment: <ex_label> ). Empty for zero-shot.
    # 3. finish with the target:  Sentence: "<sentence>"  /  Sentiment:
    # 4. return it as one string (join your lines with '\n')
    instruction = "A Financial Analyst rates the sentiment of a sentence from a company's report as negative, neutral, or positive. How would a Financial Analyst rate the financial sentiment of this sentence? Reply with one word: "
    for ex_sentence, ex_label in examples:
        instruction += f"\nSentence: {ex_sentence}\nSentiment: {ex_label}"
    instruction += f"\nSentence: {sentence}\nSentiment:"
    return instruction
    ### END YOUR CODE

### See your prompt + a real reply
Run after filling in `build_prompt`. The raw reply is what your parser (next) has to handle.

In [10]:
_demo = pb_eval[0]
print('=== PROMPT (zero-shot) ===')
print(build_prompt(_demo['sentence']))
print('\n=== MODEL REPLY ===')
print(repr(generate_reply(build_prompt(_demo['sentence']))))
print('\ntrue label:', LABELS[_demo['label']])

=== PROMPT (zero-shot) ===
A Financial Analyst rates the sentiment of a sentence from a company's report as negative, neutral, or positive. How would a Financial Analyst rate the financial sentiment of this sentence? Reply with one word: 
Sentence: The company has an annual turnover of EUR32 .8 m.
Sentiment:

=== MODEL REPLY ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


'Neutral'

true label: neutral


## 6. Parse the reply  &larr; **your code**

`parse_label(reply)` turns free text into a class index `0/1/2`, or `None` if unclear. Replies vary — `'positive'`, `'Positive.'`, `'The sentiment is positive'`. Robust approach: lowercase, then return whichever label word appears **first**. Returning `None` on no-match lets you *count* failures instead of guessing.

In [11]:
def parse_label(reply):
    """Map a free-text reply to a class index 0/1/2, or None if no label word is found."""
    ### YOUR CODE HERE
    # 1. lowercase the reply
    # 2. for each label word in LABEL_TO_IDX, note its position (text.find(word)) if it appears
    # 3. if none appear, return None; else return the index of the EARLIEST-appearing word
    reply = reply.lower()

    result = float('inf')
    return_label = None
    for sentiment, label in LABEL_TO_IDX.items():
      position = reply.find(sentiment)
      if position > -1 and position < result:
        result = position
        return_label = label
        if result == 0:
          break
    return return_label
    ### END YOUR CODE

### Test your parser
It should handle the easy cases and return `None` on the last one.

In [12]:
for _s in ['positive', 'Positive.', 'The sentiment is negative', 'Neutral', 'I would say mixed']:
    print(f'{_s!r:40s} -> {parse_label(_s)}')

'positive'                               -> 2
'Positive.'                              -> 2
'The sentiment is negative'              -> 0
'Neutral'                                -> 1
'I would say mixed'                      -> None


## 7. The evaluation loop  &larr; **your code**

`evaluate(dataset, examples=(), label='zero-shot', prompt_fn=build_prompt)` runs the pipeline over the eval set. Note the new `prompt_fn` argument: instead of hard-coding `build_prompt`, the loop calls **whatever prompt builder you pass in** — that's what lets you A/B two prompt designs in Section 9. For each row: `prompt_fn(sentence, examples)` → `generate_reply` → `parse_label`, then tally. Count **unparsed** replies (`pred is None`) separately.

In [20]:
def evaluate(dataset, examples=(), label='zero-shot', prompt_fn=build_prompt):
    """Classify each sentence via the LLM and report accuracy.

    - for each row: prompt = prompt_fn(sentence, examples); reply = generate_reply(prompt);
      pred = parse_label(reply)
    - count correct (pred == true label) and unparsed (pred is None) separately
    - print accuracy + #unparsed; return accuracy %
    """
    ### YOUR CODE HERE
    # loop over dataset (each `ex` is a dict with 'sentence' and 'label'); use prompt_fn (NOT a
    # hard-coded build_prompt) so Section 9 can swap in a second prompt. tally correct/unparsed/total;
    # print and return accuracy %.
    incorrect = []
    correct = 0
    unparsed = 0
    total = 0
    for ex in dataset:
      total += 1
      prompt = prompt_fn(ex['sentence'], examples)
      reply = generate_reply(prompt)
      pred = parse_label(reply)
      if pred == ex['label']:
        correct += 1
      elif pred is None:
        unparsed += 1
      elif pred != ex['label']:
         incorrect.append((ex['sentence'], LABELS[pred], LABELS[ex['label']]))
      total += 1
    accuracy = correct / total
    print(f'Accuracy = {accuracy:.1%}')
    print(f'Unparsed = {unparsed} ({(unparsed / total):.1%})')
    return accuracy * 100, incorrect
    ### END YOUR CODE

## 8. Predict, then run zero-shot & few-shot

#### ✍️ Predict before you run
- Zero-shot accuracy guess: _…%_  Few-shot guess: _…%_
- Will few-shot help? Why? _…_
- How big do you expect the gap to be vs. your fine-tuned FinBERT (~97%)? _…_

In [15]:
acc_zero, _ = evaluate(pb_eval, examples=(), label='zero-shot')
acc_few, _  = evaluate(pb_eval, examples=FEWSHOT_EXAMPLES, label='few-shot')
print(f'\nfew-shot lift over zero-shot: {acc_few - acc_zero:+.1f} pts')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Accuracy = 47.0%
Unparsed = 0 (0.0%)
Accuracy = 48.0%
Unparsed = 0 (0.0%)

few-shot lift over zero-shot: +1.0 pts


#### ✍️ Reflect — why did few-shot (not) help?
- Did few-shot raise accuracy, lower the `unparsed` count, both, or neither? _…_
- What two things do in-context examples give the model? _…_
- If few-shot barely moved accuracy, what does that tell you about your zero-shot prompt? _…_

## 9. Experiment — does the *wording* matter?  &larr; **your code**

This is the instrumented heart of prompt engineering. Write a **second** prompt design, `build_prompt_v2`, that differs from your first on purpose — e.g. terser, a different label menu format (`-> negative`), a stricter `Answer with exactly one word and nothing else`, or a different cue. Then the given cell runs the **same eval** with each builder and compares accuracy **and** the unparsed count. The lesson is to *see* that two reasonable prompts can score differently — wording is a lever, not a detail.

In [28]:
def build_prompt_v2(sentence, examples=()):
  """A SECOND prompt design to A/B against build_prompt (deliberately different wording/format)."""
  ### YOUR CODE HERE
  # Write a DIFFERENT prompt than build_prompt -- change the instruction wording and/or the
  # example/answer format (e.g. terser, a 'word => label' format, a stricter 'one word only').
  # Same signature: return the user-message string, handling the few-shot `examples` too.
  instruction = "You are a Financial Analyst whose task is to describe the sentiment of a sentence about a company from their financial report. Sentences can be positive (positive outlook, increased profits), neutral (facts), or negative (losses, less growth)."
  if examples:
    instruction += "\nHere are examples:"
  for ex_sentence, ex_label in examples:
    instruction += f"\nSentence: {ex_sentence}\nSentiment: {ex_label}"
  instruction += f"\n Reply with one word - positive, neutral, or negative. Here is the sentence:\nSentence: {sentence}\nSentiment: "
  return instruction

In [29]:
print('Prompt v1 (few-shot):')
acc_v1, incorrect_v1 = evaluate(pb_eval, examples=FEWSHOT_EXAMPLES, label='v1', prompt_fn=build_prompt)
print('Prompt v2 (few-shot):')
acc_v2, incorrect_v2 = evaluate(pb_eval, examples=FEWSHOT_EXAMPLES, label='v2', prompt_fn=build_prompt_v2)
print(f'\nwording effect (v2 - v1): {acc_v2 - acc_v1:+.1f} pts   (also compare the unparsed counts above)')

for wrong in incorrect_v2:
  print(wrong)

Prompt v2 (few-shot):


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Accuracy = 48.0%
Unparsed = 0 (0.0%)

wording effect (v2 - v1): +0.0 pts   (also compare the unparsed counts above)
('The loss for the third quarter of 2007 was EUR 0.3 mn smaller than the loss of the second quarter of 2007 .', 'neutral', 'positive')
('MANAVIGATOR-September 7 , 2010-Kemira unveils Indian JV with IVRCL Finnish chemicals group Kemira ( HEL : KRA1V ) on Tuesday announced it has inked a deal to form a joint venture in India with local construction firm IVRCL Infrastructure and Projects Ltd ( BOM :530773 ) .', 'neutral', 'positive')
('BioTie North-American licensing partner Somaxon Pharmaceuticals started a phase II-III clinical study in patients suffering from pathological gambling and a pilot phase II study in nicotine addiction smoking cessation .', 'positive', 'neutral')
("The company 's operating income ( EBIT ) totalled EUR 0.0 mn , up from EUR -0.3 mn year-on-year .", 'neutral', 'positive')
('In 2008 Stockmann earned 3.398 million lats in profit on 48.012 million lat

#### ✍️ Reflect — wording as a lever
- Which prompt scored higher, and by how much? Did the `unparsed` counts differ? Prompt 2, just barely
- If v2 cut the unparsed count, was the accuracy gain from better *judgment* or just better *formatting*? No difference
- What one change would you try next to push it further? We can try to see the what sentences are confusing the model and provide more context within the prompt

## 10. The payoff — is fine-tuning worth it?
This is the whole reason Part 5 exists. Put your numbers next to the trained models.

#### ✍️ The verdict — record and decide
- Your numbers: zero-shot _…%_, few-shot _…%_.
- Gap vs. fine-tuned FinBERT (~97%) and GPT-2 (82–86%): _…_
- Given prompting needs **no labels / no training / no GPU**, when would you ship the prompt instead of fine-tuning? When not? _…_
- Add the zero-shot and few-shot rows to the `Choosing_a_Training_Method` Part B table.

## 11. Recap & next

You built the no-training baseline by hand: a prompt, a parser, an eval loop — and you **measured** two levers (few-shot vs zero-shot, and v1 vs v2 wording) instead of taking them on faith. Watch the `unparsed` count as a health check on prompt/parser quality, and remember `neutral` is the hard class (hedging).

**Things to try:** more shots (two per class); ask for a one-word answer *then* a short reason (often improves the label); raise `EVAL_N` toward the full 453.

**Next:** Part 6 — **synthetic data generation**: prompt this same model to *generate* labeled financial sentences, add them to the Part 2/3 training set, and measure the accuracy change (Track B feeding Track A).